In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict

import torch
import torch.nn as nn


@dataclass
class LossWeights:
    w_data_u: float = 1.0
    w_data_v: float = 1.0
    w_data_p: float = 2.0

    w_pde: float = 1.0
    w_cont: float = 1.0
    w_mom: float = 1.0

    w_inlet_u: float = 2.0
    w_inlet_v: float = 2.0
    w_inlet_p: float = 1.0

    w_outlet_p: float = 5.0

    w_wall_u: float = 1.0
    w_wall_v: float = 1.0

    w_body_u: float = 1.0
    w_body_v: float = 1.0
    w_body_p: float = 5.0

    w_probe_dp: float = 10.0
    w_grad_p: float = 1.0

    w_pde_near_body: float = 4.0
    near_body_k: float = 5.0

    phi_min: float = 0.0
    zero_mean_data_p: bool = False


def _grad(y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return torch.autograd.grad(
        y,
        x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]


def mse(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    if a.numel() == 0:
        return torch.zeros((), device=a.device, dtype=a.dtype)
    return torch.mean((a - b) ** 2)


def mse_norm(pred: torch.Tensor, targ: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    if pred.numel() == 0:
        return torch.zeros((), device=pred.device, dtype=pred.dtype)
    scale = torch.sqrt(torch.mean(targ**2) + eps)
    return torch.mean(((pred - targ) / scale) ** 2)


def _filter_fluid(
    X: torch.Tensor,
    Y: torch.Tensor | None,
    phi_min: float,
):
    if X.numel() == 0:
        return X, Y

    mask = X[:, 3] > float(phi_min)
    Xf = X[mask]

    if Y is None:
        return Xf, None

    return Xf, Y[mask]


def _zero_mean_if_enabled(pred_p: torch.Tensor, targ_p: torch.Tensor, enabled: bool):
    if not enabled:
        return pred_p, targ_p
    return pred_p - torch.mean(pred_p), targ_p - torch.mean(targ_p)


def ns_residuals(
    model: nn.Module,
    X: torch.Tensor,
    Re_max: float,
):
    X = X.requires_grad_(True)
    Y = model(X)

    u = Y[:, 0:1]
    v = Y[:, 1:2]
    p = Y[:, 2:3]

    du = _grad(u, X)
    dv = _grad(v, X)
    dp = _grad(p, X)

    u_x, u_y = du[:, 0:1], du[:, 1:2]
    v_x, v_y = dv[:, 0:1], dv[:, 1:2]
    p_x, p_y = dp[:, 0:1], dp[:, 1:2]

    u_xx = _grad(u_x, X)[:, 0:1]
    u_yy = _grad(u_y, X)[:, 1:2]
    v_xx = _grad(v_x, X)[:, 0:1]
    v_yy = _grad(v_y, X)[:, 1:2]

    Re = X[:, 2:3] * float(Re_max)
    Re = torch.clamp(Re, min=1e-6)

    r_cont = u_x + v_y
    r_u = (u * u_x + v * u_y) + p_x - (1.0 / Re) * (u_xx + u_yy)
    r_v = (u * v_x + v * v_y) + p_y - (1.0 / Re) * (v_xx + v_yy)

    phi = X[:, 3:4]
    phi_pos = torch.clamp(phi, min=0.0)

    return r_cont, r_u, r_v, phi_pos


def _optional_probe_dp_loss(model: nn.Module, batch, dev, dtype):
    z = torch.zeros((), device=dev, dtype=dtype)

    probes = getattr(batch, "probes", None)
    if probes is None:
        return z, torch.tensor(float("nan"), device=dev, dtype=dtype)

    if "front_X" not in probes or "rear_X" not in probes:
        return z, torch.tensor(float("nan"), device=dev, dtype=dtype)

    target = float((getattr(batch, "targets", {}) or {}).get("deltap_nd", float("nan")))
    if not torch.isfinite(torch.tensor(target)):
        return z, torch.tensor(float("nan"), device=dev, dtype=dtype)

    Xf = probes["front_X"]
    Xr = probes["rear_X"]

    Xf = torch.as_tensor(Xf, device=dev, dtype=dtype)
    Xr = torch.as_tensor(Xr, device=dev, dtype=dtype)

    if Xf.ndim == 1:
        Xf = Xf.unsqueeze(0)
    if Xr.ndim == 1:
        Xr = Xr.unsqueeze(0)

    pf = model(Xf)[:, 2:3].mean()
    pr = model(Xr)[:, 2:3].mean()
    pred_dp = pf - pr

    targ = torch.tensor(target, device=dev, dtype=dtype)
    loss = ((pred_dp - targ) / (torch.abs(targ) + 1e-6)) ** 2
    return loss.reshape(()), pred_dp.reshape(())


def _optional_grad_p_loss(model: nn.Module, batch, dev, dtype):
    z = torch.zeros((), device=dev, dtype=dtype)

    extra = getattr(batch, "extra", None)
    if extra is None:
        return z, z, z

    if "X" not in extra or "dpdx" not in extra or "dpdy" not in extra:
        return z, z, z

    Xg = torch.as_tensor(extra["X"], device=dev, dtype=dtype)
    tgx = torch.as_tensor(extra["dpdx"], device=dev, dtype=dtype)
    tgy = torch.as_tensor(extra["dpdy"], device=dev, dtype=dtype)

    if Xg.numel() == 0:
        return z, z, z

    Xg, _ = _filter_fluid(Xg, None, 0.0)
    if Xg.numel() == 0:
        return z, z, z

    Xg = Xg.clone().detach().requires_grad_(True)
    pred = model(Xg)[:, 2:3]
    dp = _grad(pred, Xg)

    pdx = dp[:, 0:1]
    pdy = dp[:, 1:2]

    tgx = tgx[: len(Xg)].reshape(-1, 1)
    tgy = tgy[: len(Xg)].reshape(-1, 1)

    Lx = mse_norm(pdx, tgx)
    Ly = mse_norm(pdy, tgy)
    return Lx + Ly, Lx, Ly


def compute_losses(
    model: nn.Module,
    batch,
    Re_max: float,
    weights: LossWeights,
    outlet_p_target: float = 0.0,
) -> Dict[str, torch.Tensor]:
    dev = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    z = torch.zeros((), device=dev, dtype=dtype)

    L_data_u = z
    L_data_v = z
    L_data_p = z

    Xd = batch.X_data
    Yd = batch.Y_data

    if bool(getattr(batch, "has_data", False)) and Xd.numel() > 0:
        Xd, Yd = _filter_fluid(Xd, Yd, weights.phi_min)

        if Xd.numel() > 0:
            pred_d = model(Xd)

            pred_p = pred_d[:, 2:3]
            targ_p = Yd[:, 2:3]
            pred_p, targ_p = _zero_mean_if_enabled(pred_p, targ_p, weights.zero_mean_data_p)

            L_data_u = mse_norm(pred_d[:, 0:1], Yd[:, 0:1])
            L_data_v = mse_norm(pred_d[:, 1:2], Yd[:, 1:2])
            L_data_p = mse_norm(pred_p, targ_p)

    L_data = (
        weights.w_data_u * L_data_u +
        weights.w_data_v * L_data_v +
        weights.w_data_p * L_data_p
    )

    Xp, _ = _filter_fluid(batch.X_pde, None, weights.phi_min)

    L_cont = z
    L_mom = z
    L_pde = z

    if Xp.numel() > 0:
        r_cont, r_u, r_v, phi_pos = ns_residuals(model, Xp, Re_max=Re_max)

        cont_sq = r_cont ** 2
        mom_sq = r_u ** 2 + r_v ** 2

        alpha = float(weights.w_pde_near_body)
        if abs(alpha - 1.0) > 1e-12:
            w = torch.exp(-float(weights.near_body_k) * phi_pos)
            w = 1.0 + (alpha - 1.0) * w
            w = w / (torch.mean(w) + 1e-12)

            L_cont = torch.mean(w * cont_sq)
            L_mom = torch.mean(w * mom_sq)
        else:
            L_cont = torch.mean(cont_sq)
            L_mom = torch.mean(mom_sq)

        L_pde = weights.w_cont * L_cont + weights.w_mom * L_mom

    L_inlet_u = z
    L_inlet_v = z
    L_inlet_p = z

    L_outlet_p = z

    L_wall_u = z
    L_wall_v = z

    L_body_u = z
    L_body_v = z
    L_body_p = z

    inlet = batch.bc.get("inlet", None)
    if inlet is not None and inlet["X"].numel() > 0 and bool(inlet["has_Y"]):
        Xi = inlet["X"]
        Yi = inlet["Y"]
        pred_i = model(Xi)

        L_inlet_u = mse_norm(pred_i[:, 0:1], Yi[:, 0:1])
        L_inlet_v = mse_norm(pred_i[:, 1:2], Yi[:, 1:2])
        L_inlet_p = mse_norm(pred_i[:, 2:3], Yi[:, 2:3])

    outlet = batch.bc.get("outlet", None)
    if outlet is not None and outlet["X"].numel() > 0:
        Xo = outlet["X"]
        pred_o = model(Xo)[:, 2:3]

        if bool(outlet["has_Y"]):
            Yo = outlet["Y"][:, 2:3]
            L_outlet_p = mse_norm(pred_o, Yo)
        else:
            tgt = torch.full_like(pred_o, float(outlet_p_target))
            L_outlet_p = mse(pred_o, tgt)

    wall = batch.bc.get("wall", None)
    if wall is not None and wall["X"].numel() > 0 and bool(wall["has_Y"]):
        Xw = wall["X"]
        Yw = wall["Y"]
        pred_w = model(Xw)

        if Yw.shape[1] >= 2:
            L_wall_u = mse_norm(pred_w[:, 0:1], Yw[:, 0:1])
            L_wall_v = mse_norm(pred_w[:, 1:2], Yw[:, 1:2])

    body = batch.bc.get("body", None)
    if body is not None and body["X"].numel() > 0 and bool(body["has_Y"]):
        Xb = body["X"]
        Yb = body["Y"]
        pred_b = model(Xb)

        if Yb.shape[1] >= 2:
            L_body_u = mse_norm(pred_b[:, 0:1], Yb[:, 0:1])
            L_body_v = mse_norm(pred_b[:, 1:2], Yb[:, 1:2])

        if Yb.shape[1] >= 3:
            L_body_p = mse_norm(pred_b[:, 2:3], Yb[:, 2:3])

    L_bc = (
        weights.w_inlet_u * L_inlet_u +
        weights.w_inlet_v * L_inlet_v +
        weights.w_inlet_p * L_inlet_p +
        weights.w_outlet_p * L_outlet_p +
        weights.w_wall_u * L_wall_u +
        weights.w_wall_v * L_wall_v +
        weights.w_body_u * L_body_u +
        weights.w_body_v * L_body_v +
        weights.w_body_p * L_body_p
    )

    L_probe_dp, pred_deltap_nd = _optional_probe_dp_loss(model, batch, dev, dtype)
    L_grad_p, L_grad_px, L_grad_py = _optional_grad_p_loss(model, batch, dev, dtype)

    L_total = (
        L_data +
        weights.w_pde * L_pde +
        L_bc +
        weights.w_probe_dp * L_probe_dp +
        weights.w_grad_p * L_grad_p
    )

    batch_targets = getattr(batch, "targets", {}) or {}

    target_Cd = torch.tensor(float(batch_targets.get("Cd", float("nan"))), device=dev, dtype=dtype)
    target_Cl = torch.tensor(float(batch_targets.get("Cl", float("nan"))), device=dev, dtype=dtype)
    target_deltap_nd = torch.tensor(float(batch_targets.get("deltap_nd", float("nan"))), device=dev, dtype=dtype)
    target_La_nd = torch.tensor(float(batch_targets.get("La_nd", float("nan"))), device=dev, dtype=dtype)

    return {
        "L_total": L_total,

        "L_data": L_data,
        "L_data_u": L_data_u,
        "L_data_v": L_data_v,
        "L_data_p": L_data_p,

        "L_pde": L_pde,
        "L_cont": L_cont,
        "L_mom": L_mom,

        "L_bc": L_bc,
        "L_inlet_u": L_inlet_u,
        "L_inlet_v": L_inlet_v,
        "L_inlet_p": L_inlet_p,
        "L_outlet_p": L_outlet_p,
        "L_body_p": L_body_p,

        "L_wall_u": L_wall_u,
        "L_wall_v": L_wall_v,
        "L_body_u": L_body_u,
        "L_body_v": L_body_v,

        "L_probe_dp": L_probe_dp,
        "pred_deltap_nd": pred_deltap_nd,

        "L_grad_p": L_grad_p,
        "L_grad_px": L_grad_px,
        "L_grad_py": L_grad_py,

        "target_Cd": target_Cd,
        "target_Cl": target_Cl,
        "target_deltap_nd": target_deltap_nd,
        "target_La_nd": target_La_nd,
    }